In [0]:
import mlflow
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score, roc_auc_score
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "fraud_lakehouse")
CATALOG = dbutils.widgets.get("catalog")
MODEL_NAME = f"{CATALOG}.ml.fraud_detector"
FEATURES = ["amount", "distance_from_home_km", "hour_of_day", "is_night", "is_online",
            "is_high_risk_category", "txn_count_1h", "amount_sum_1h", "amount_vs_30d_avg"]

features = spark.table(f"{CATALOG}.gold.fraud_features")
display(
    features.groupBy("is_fraud").agg(
        F.count("*").alias("transactions"),
        F.round(F.avg("txn_count_1h"), 2).alias("avg_txn_count_1h"),
        F.round(F.avg("amount_vs_30d_avg"), 2).alias("avg_amount_vs_30d"),
    )
)

In [0]:
df = features.select("date_key", *FEATURES, "is_fraud").toPandas()

cutoff = df["date_key"].quantile(0.8)
train, test = df[df["date_key"] <= cutoff], df[df["date_key"] > cutoff]
print(f"train={len(train):,}  test={len(test):,}  cutoff={int(cutoff)}")

In [0]:
user = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/fraud_detector_experiment")
mlflow.set_registry_uri("databricks-uc")

params = {"max_iter": 200, "learning_rate": 0.1, "max_depth": 6, "class_weight": "balanced"}

with mlflow.start_run():
    model = HistGradientBoostingClassifier(**params, random_state=42).fit(train[FEATURES], train["is_fraud"])
    proba = model.predict_proba(test[FEATURES])[:, 1]
    pred = proba >= 0.5
    rule = (test["amount_vs_30d_avg"] > 3) | (test["txn_count_1h"] >= 3)  # simple rules-engine baseline

    metrics = {
        "pr_auc": average_precision_score(test["is_fraud"], proba),
        "roc_auc": roc_auc_score(test["is_fraud"], proba),
        "precision": precision_score(test["is_fraud"], pred),
        "recall": recall_score(test["is_fraud"], pred),
        "rule_precision": precision_score(test["is_fraud"], rule),
        "rule_recall": recall_score(test["is_fraud"], rule),
    }
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    info = mlflow.sklearn.log_model(
        model, name="model",
        signature=infer_signature(test[FEATURES], proba),
        input_example=test[FEATURES].head(5),
        registered_model_name=MODEL_NAME,
    )

print({k: round(v, 3) for k, v in metrics.items()})

In [0]:
client = MlflowClient()
new_version = info.registered_model_version

try:
    champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
    best = client.get_run(champion.run_id).data.metrics["pr_auc"]
except mlflow.exceptions.MlflowException:
    best = -1.0  # first run, no champion yet

if metrics["pr_auc"] > best:
    client.set_registered_model_alias(MODEL_NAME, "champion", new_version)
    print(f"v{new_version} is the new champion (pr_auc {metrics['pr_auc']:.3f})")
else:
    print(f"champion kept (pr_auc {best:.3f} >= {metrics['pr_auc']:.3f})")